# Exploratory Data Analysis (EDA) – India Housing Prices

**Objective:** Explore the supplied housing dataset, understand its structure and quality, study property-price distributions and relationships, and summarise useful patterns before using the Streamlit dashboard.

**Dataset:** `india_housing_prices(2).csv` (keep this CSV in the same folder as this notebook).

> This notebook performs descriptive analysis only. It does not establish verified market valuations or causal relationships.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the dataset

In [ ]:
DATA_FILE = "india_housing_prices(2).csv"
df = pd.read_csv(DATA_FILE)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

## 3. Understand the data

In [ ]:
df.info()

In [ ]:
display(df.describe(include="all").T)

## 4. Data-quality assessment

In [ ]:
# Missing values and their percentage
missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)

display(missing[missing["Missing_Count"] > 0])

# Duplicate records
print("Duplicate rows:", f"{df.duplicated().sum():,}")
print("Duplicate IDs:", f"{df['ID'].duplicated().sum():,}" if "ID" in df.columns else "ID column not present")

# Check data types and unique values in categorical columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
display(pd.DataFrame({
    "Data_Type": df[categorical_cols].dtypes.astype(str),
    "Unique_Values": df[categorical_cols].nunique(dropna=True)
}))

## 5. Prepare analysis columns

In [ ]:
# Work on a copy. Convert key numeric fields safely; original df remains unchanged.
eda = df.copy()
numeric_candidates = [
    "BHK", "Size_in_SqFt", "Price_in_Lakhs", "Price_per_SqFt",
    "Year_Built", "Floor_No", "Total_Floors", "Age_of_Property",
    "Nearby_Schools", "Nearby_Hospitals"
]
for col in numeric_candidates:
    if col in eda.columns:
        eda[col] = pd.to_numeric(eda[col], errors="coerce")

# Standardise whitespace in text columns without changing category meaning.
for col in eda.select_dtypes(include="object").columns:
    eda[col] = eda[col].astype("string").str.strip()

display(eda[numeric_candidates].describe().T)

## 6. Univariate analysis – individual variables

In [ ]:
# Distribution of listing prices
if "Price_in_Lakhs" in eda.columns:
    values = eda["Price_in_Lakhs"].dropna()
    plt.figure(figsize=(9, 5))
    plt.hist(values, bins=50)
    plt.title("Distribution of Property Prices")
    plt.xlabel("Price (₹ lakhs)")
    plt.ylabel("Number of properties")
    plt.tight_layout()
    plt.show()

# Distribution of property sizes
if "Size_in_SqFt" in eda.columns:
    values = eda["Size_in_SqFt"].dropna()
    plt.figure(figsize=(9, 5))
    plt.hist(values, bins=50)
    plt.title("Distribution of Property Sizes")
    plt.xlabel("Size (sq. ft.)")
    plt.ylabel("Number of properties")
    plt.tight_layout()
    plt.show()

# Counts by property type and BHK
for col in ["Property_Type", "BHK", "Furnished_Status"]:
    if col in eda.columns:
        counts = eda[col].value_counts(dropna=False).head(15)
        plt.figure(figsize=(9, 4))
        counts.plot(kind="bar")
        plt.title(f"Number of Properties by {col}")
        plt.xlabel(col)
        plt.ylabel("Number of properties")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()

## 7. Bivariate analysis – relationships between variables

In [ ]:
# Property size versus price
if {"Size_in_SqFt", "Price_in_Lakhs"}.issubset(eda.columns):
    sample = eda[["Size_in_SqFt", "Price_in_Lakhs"]].dropna()
    if len(sample) > 20000:
        sample = sample.sample(20000, random_state=42)
    plt.figure(figsize=(8, 5))
    plt.scatter(sample["Size_in_SqFt"], sample["Price_in_Lakhs"], alpha=0.25)
    plt.title("Property Size vs Price")
    plt.xlabel("Size (sq. ft.)")
    plt.ylabel("Price (₹ lakhs)")
    plt.tight_layout()
    plt.show()

# Median price by BHK
if {"BHK", "Price_in_Lakhs"}.issubset(eda.columns):
    bhk_price = eda.groupby("BHK")["Price_in_Lakhs"].median().sort_index()
    display(bhk_price.rename("Median_Price_Lakhs").to_frame())
    bhk_price.plot(kind="bar", figsize=(8, 4), title="Median Property Price by BHK")
    plt.xlabel("BHK")
    plt.ylabel("Median price (₹ lakhs)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Median price by property type
if {"Property_Type", "Price_in_Lakhs"}.issubset(eda.columns):
    type_price = (eda.groupby("Property_Type")["Price_in_Lakhs"]
                  .median().sort_values(ascending=False))
    display(type_price.rename("Median_Price_Lakhs").to_frame())
    type_price.plot(kind="bar", figsize=(8, 4), title="Median Property Price by Type")
    plt.xlabel("Property type")
    plt.ylabel("Median price (₹ lakhs)")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 8. Location analysis

In [ ]:
if {"City", "Price_in_Lakhs"}.issubset(eda.columns):
    city_summary = (eda.groupby("City")
                    .agg(Property_Count=("Price_in_Lakhs", "size"),
                         Median_Price_Lakhs=("Price_in_Lakhs", "median"),
                         Average_Price_Lakhs=("Price_in_Lakhs", "mean"))
                    .sort_values("Median_Price_Lakhs", ascending=False))
    display(city_summary.head(15))

    city_summary.head(15)["Median_Price_Lakhs"].sort_values().plot(
        kind="barh", figsize=(9, 6), title="15 Cities with Highest Median Listing Prices"
    )
    plt.xlabel("Median price (₹ lakhs)")
    plt.tight_layout()
    plt.show()

## 9. Outlier exploration using the IQR method

In [ ]:
# IQR flags potential outliers; it does not automatically remove them.
def iqr_outlier_summary(series):
    x = pd.to_numeric(series, errors="coerce").dropna()
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((x < lower) | (x > upper)).sum()
    return pd.Series({
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower_Bound": lower, "Upper_Bound": upper,
        "Potential_Outliers": int(count),
        "Outlier_Percent": round(count / len(x) * 100, 2) if len(x) else np.nan
    })

for col in ["Price_in_Lakhs", "Size_in_SqFt", "Price_per_SqFt"]:
    if col in eda.columns:
        print(f"Potential outliers in {col}")
        display(iqr_outlier_summary(eda[col]).to_frame("Value"))

## 10. Correlation analysis

In [ ]:
numeric_df = eda.select_dtypes(include=np.number)
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr(numeric_only=True)
    display(corr.round(2))

    # Matplotlib heatmap without an additional plotting dependency
    fig, ax = plt.subplots(figsize=(11, 8))
    image = ax.imshow(corr, aspect="auto", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.columns)
    ax.set_title("Correlation Matrix of Numeric Variables")
    fig.colorbar(image, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for correlation analysis.")

## 11. Key summary and interpretation

In [ ]:
print("EDA SUMMARY")
print("=" * 60)
print(f"Total records: {len(eda):,}")
print(f"Total columns: {eda.shape[1]:,}")
print(f"Duplicate rows: {eda.duplicated().sum():,}")
print(f"Missing cells: {int(eda.isna().sum().sum()):,}")

if "Price_in_Lakhs" in eda.columns:
    print(f"Average listed price: ₹{eda['Price_in_Lakhs'].mean():,.2f} lakhs")
    print(f"Median listed price: ₹{eda['Price_in_Lakhs'].median():,.2f} lakhs")

if "Size_in_SqFt" in eda.columns:
    print(f"Average property size: {eda['Size_in_SqFt'].mean():,.2f} sq. ft.")

if "State" in eda.columns:
    print(f"States represented: {eda['State'].nunique(dropna=True)}")
if "City" in eda.columns:
    print(f"Cities represented: {eda['City'].nunique(dropna=True)}")

### How to write conclusions after running the notebook

Use the actual output above to write 4–6 evidence-based findings. For example, comment on the distribution of prices, how median prices vary by city or property type, whether size and price move together, and whether missing values or potential outliers need attention.

Do not state that a relationship is causal. A correlation or chart shows an association in this dataset, not proof that one factor causes another.